# Titanic Machine Learning Project



### 1. Setup and Data Loading


**1.1 Import libraries**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Reproductibility
np.random.seed(42)


**1.2 Import the titanic data**

In [2]:
DATA_PATH = Path("datasets/titanic")
train_df = pd.read_csv(DATA_PATH / "train.csv")
test_df = pd.read_csv(DATA_PATH / "test.csv")

In [3]:
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
test_df.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


Both dataset loaded successfully. Let's find out data set's structure  before deep dive deep Exploratory Data Analysis

**1.3 Short Observations**

In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


As we seen above, Cabin has a lot of null values and also Age does have NULL values this is our first observation.

In [6]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    object 
 3   Sex          418 non-null    object 
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    object 
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     object 
 10  Embarked     418 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 36.1+ KB


Same with the above almost same proportion of `Age and Cabin` is missing in the test dataset.

In [7]:
train_df["Survived"].value_counts(normalize = True)
# normalize = True does convert raw counts into relative frequencies (proportions that sum to 1.0)

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

In [8]:
print(f"Shape of Train dataset {train_df.shape}")
print(f"Shape of Test dataset {test_df.shape}")

Shape of Train dataset (891, 12)
Shape of Test dataset (418, 11)


In [11]:
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


`PassengerID` - Drop it. It's an arbitrary index, no signal or correlation with the dataset.


`Name` : The full name has no signal. But the **title embedded in the name does**. 'Mr' , 'Miss', 'Master', 'Dr' ,'Rev' etc. Think about what a title encodes. It's a proxy for sex, age, band, marital status, and social rank. 'Master' specifically means a young boy, which is real signal given 'women and children first' Extracting **Title** from **Name** with a custom transformer is one of the highest=value engineered features on this dataset. So don't use Name raw, but don't drop it before extracting Title.


`Ticket`: It's messy, shared ticket numbers indicate groups travelling together. It's a lower value-per-effort. Reasonable to investigate briefly in EDA and drop if it looks too nosiy.

`Cabin`: Here's the nuance 77% missing means we can't imput it sensibly. But before dropping consider: Is the missingness itself informative ? A common trick is to engineer a binary HasCabin feature (1 if cabin recored, 0 if note). The hyphotehesis: cabin was more likely recorded for higher-class passengers, so missingess correlated with class adn possibly survivalc. You could also extract just the deck letter (C85 -> C)


## Step 2: Exploratory Data Analysis



#### 2.1 Categorical Features versus Survival.

For `Sex`, `Pclass`, `Embarked`, group by each and look at the mean of Survived (since it's 0/1 , the mean is the survival rate) You're checking: does this feature separate survivors from non-survivors ?

In [22]:
train_df["Survived"].mean()

0.3838383838383838

In [23]:
train_df.groupby(by = 'Sex')['Survived'].mean()

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

`Sex` female 74% survival versus male 19%. That's a massive gap. This is the single strongest predictor in the whole dataset. 'Women and children first' shows up directly in the numbers. Definetly keep it.

In [25]:
train_df.groupby(by = 'Pclass')['Survived'].mean()

Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

`Pclass` 1st class 63%, 2nd 47%, 3rd 24%. Clear monotonic gradient. Higher class, higher survival. Strong feature, keep. Note it's ordinal (1<2<3 means something), which is a small decision for later. one-hot encode it, or treat as ordinal ? Either works; one-hot is the safe default.

In [26]:
train_df.groupby(by = 'Embarked')["Survived"].mean()

Embarked
C    0.553571
Q    0.389610
S    0.336957
Name: Survived, dtype: float64

`Embarked` C 55%, Q 39%, S 34%. There's a real spread, so it has some signal. But be a little skeptical. Embarked is probaly correlated with Pclass (Cherbourg likely had more 1st-class passengers) so partt o this 'signal' may just be Pclass leaking through. Keep it, but it's a weaker feature than the first two.

### 2.2 Numerical features versus Survival

for `Age` and `Fare` those are continous variables and we can't just groupby a continous variable. There are two options bin them (e.g. `pd.cut` Age into child/teen/adult/senior) then compare survival per bin. or plot overlapping histograms one for survivors one forf non-survivors and see if the distribution is differ.



In [35]:
train_df.groupby(by = pd.cut(train_df["Age"], bins = [0,12,18,35,60,100], labels= ["Child", "Teen","Adult","Senior","Elder"]), observed=True)["Survived"].mean() 

Age
Child     0.579710
Teen      0.428571
Adult     0.382682
Senior    0.400000
Elder     0.227273
Name: Survived, dtype: float64

Age does carry signal, but it's concentrated at the extremes, being a child helps a lot, being elderly hurts. The middle three bands are clustered around 38% which is uninformative.<br>
This tells something useful for Step3: the relationship between Age and survival is non-linear (high at low age, drops off, low at high age) A linear model like `LogisticRegression` will struggle to capture 'extremes matter, middle doesn't' from raw Age it can only fit a straight line through it. Two implictions:

- Tree-based models (RandomForest, GradientBoosting) will handle raw Age fine. They split on thresholds naturally.
- If you want a linear model to use Age well, feedin it the banded version (or an IsChild binary flag) often works bettern than raw Age. This is the same lesson as Cabin earlier. the engineered form can beat the raw form. 

In [ ]:
train_df.groupby(by = pd.qcut(train_df["Fare"], q = 4, labels=["Low-end", 'Medium', 'High', 'First Class']), observed=True)["Survived"].mean()

Fare
Low-end        0.197309
Medium         0.303571
High           0.454955
First Class    0.581081
Name: Survived, dtype: float64

A clear, monotonic climb. Higher fare, higher survival. no surprises or kinks.

So Fare has strong signal but here's the thing to notice this curve almost identical to the Pclass curve we found earlier. That's not a coincidence. Fare and Pclass are basically measuring the same underlying thing. 

This is **multicollinearity**. and it's worth flagging now for step 3:


`Cabin` Let's look at the cabin feature transform it to are people have Cabin or Not (we're knowing from the dataset if it's null there is no cabin if it's a value then it has a cabin)

In [59]:
def has_cabin(value):
    if pd.isna(value):
        return "No"
    return "Yes"
train_df["HasCabin"] = train_df["Cabin"].apply(has_cabin)
train_df["HasCabin"]

0       No
1      Yes
2       No
3      Yes
4       No
      ... 
886     No
887    Yes
888     No
889    Yes
890     No
Name: HasCabin, Length: 891, dtype: object

In [60]:
train_df.groupby("HasCabin")["Survived"].mean()

HasCabin
No     0.299854
Yes    0.666667
Name: Survived, dtype: float64

No cabin recorded 30% of survival chance while Cabin recorded 67% survival.

In [61]:
train_df["Parch"]

0      0
1      0
2      0
3      0
4      0
      ..
886    0
887    0
888    2
889    0
890    0
Name: Parch, Length: 891, dtype: int64

`FamilySize` let's create a familySize feature depends on `SibSp` = number of siblings + Spouses  and `Parch` = number of Parents + Childrent abroad. 

In [63]:
train_df["FamilySize"] = train_df["SibSp"] + train_df["Parch"] + 1
train_df["FamilySize"]

0      2
1      2
2      1
3      2
4      1
      ..
886    1
887    1
888    4
889    1
890    1
Name: FamilySize, Length: 891, dtype: int64

In [64]:
train_df.groupby(by = "FamilySize")["Survived"].mean()

FamilySize
1     0.303538
2     0.552795
3     0.578431
4     0.724138
5     0.200000
6     0.136364
7     0.333333
8     0.000000
11    0.000000
Name: Survived, dtype: float64

In [65]:
train_df["FamilySize"].value_counts()

FamilySize
1     537
2     161
3     102
4      29
6      22
5      15
7      12
11      7
8       6
Name: count, dtype: int64

## Exploratory Data Analysis Done.

Let's decide how we'll handle the features, which features we going to drop, keep, engineer how we handle the missing values and so on.

Before moving to the next step which is Pipeline we have to locked the decisions.